# Fine-tuned Qwen3 adapters: CTC greedy vs 10-best translation

추가 학습 없이 다음 두 adapter를 평가합니다.

- English: `clean_lora_best`
- Korean: `korean_clean_lora_epoch1`
- Input: notebook 13이 저장한 동일 held-out 300개의 CTC greedy / 10-best

총 300 utterances × 2 languages × 2 conditions = 1,200 generations입니다.


In [1]:
!pip -q uninstall -y torchao
!pip -q install -U "transformers==4.57.3" "peft==0.17.1" accelerate safetensors pandas sacrebleu


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np,pandas as pd,torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer,AutoModelForCausalLM
from peft import PeftModel
from sacrebleu.metrics import CHRF,BLEU

DEVICE=torch.device('cuda');DTYPE=torch.bfloat16
ROOT=Path('/content/drive/MyDrive/S2ST')
CTC_DIR=ROOT/'baselines/qwen_token_ctc_nbest'
LORA_ROOT=ROOT/'baselines/qwen3_translation_lora'
EN_DIR=LORA_ROOT/'clean_lora_best'
KO_DIR=LORA_ROOT/'korean_clean_lora_epoch1'
OUT_DIR=LORA_ROOT/'finetuned_ctc_nbest_eval';OUT_DIR.mkdir(parents=True,exist_ok=True)
LLM_ID='Qwen/Qwen3-4B-Instruct-2507';MAX_NEW_TOKENS=128
for p in [CTC_DIR/'dev_nbest.pkl',EN_DIR,KO_DIR]:assert p.exists(),p
print(torch.cuda.get_device_name(),EN_DIR,KO_DIR)


Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


NVIDIA RTX PRO 6000 Blackwell Server Edition /content/drive/MyDrive/S2ST/baselines/qwen3_translation_lora/clean_lora_best /content/drive/MyDrive/S2ST/baselines/qwen3_translation_lora/korean_clean_lora_epoch1


## 1. Load the fixed CTC evaluation set and both adapters


In [3]:
decoded=pd.read_pickle(CTC_DIR/'dev_nbest.pkl')
assert len(decoded)==300 and decoded.sample_id.nunique()==300
required={'sample_id','source','my_text','greedy_text','nbest_texts','translation_en','translation_ko'}
assert required.issubset(decoded.columns),set(decoded.columns)

tokenizer=AutoTokenizer.from_pretrained(EN_DIR,use_fast=True)
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
base=AutoModelForCausalLM.from_pretrained(LLM_ID,dtype=DTYPE,device_map={'':'cuda:0'},attn_implementation='sdpa')
model=PeftModel.from_pretrained(base,EN_DIR,adapter_name='english',is_trainable=False)
model.load_adapter(KO_DIR,adapter_name='korean',is_trainable=False)
model.eval()
print('loaded adapters:',list(model.peft_config),'samples:',len(decoded))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

loaded adapters: ['english', 'korean'] samples: 300


## 2. Matched prompts and generation


In [4]:
def clean_prompt(text,lang):
    target='English' if lang=='en' else 'Korean'
    return f'Translate the following Myanmar text into {target}. Preserve its meaning and return only the translation.\n\nMyanmar:\n{text}'

def nbest_prompt(candidates,lang):
    target='English' if lang=='en' else 'Korean'
    numbered='\n'.join(f'{i+1}. {x}' for i,x in enumerate(candidates))
    return f"""Translate uncertain Myanmar speech recognition candidates into {target}.
All candidates describe the same utterance and are ordered by acoustic probability.
Use their shared context to recover the intended meaning. Return only the {target} translation.

Myanmar candidates:
{numbered}"""

@torch.inference_mode()
def generate(prompt,lang):
    model.set_adapter('english' if lang=='en' else 'korean')
    text=tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True)
    inp=tokenizer(text,return_tensors='pt').to(DEVICE)
    out=model.generate(**inp,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,repetition_penalty=1.05,
                       eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0,inp.input_ids.shape[1]:],skip_special_tokens=True).strip()

rows=[]
for r in tqdm(decoded.itertuples(),total=len(decoded),desc='fine-tuned greedy/N-best translation'):
    for lang,ref_col in [('en','translation_en'),('ko','translation_ko')]:
        greedy=generate(clean_prompt(str(r.greedy_text),lang),lang)
        nbest=generate(nbest_prompt(list(r.nbest_texts),lang),lang)
        rows.append({'sample_id':str(r.sample_id),'source':str(r.source),'language':lang,'reference':str(getattr(r,ref_col)),
                     'my_text':str(r.my_text),'greedy_text':str(r.greedy_text),'greedy_translation':greedy,'nbest_translation':nbest})
pred_df=pd.DataFrame(rows);pred_df.to_csv(OUT_DIR/'predictions.csv',index=False)
print('saved predictions:',len(pred_df))


fine-tuned greedy/N-best translation:   0%|          | 0/300 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


saved predictions: 600


## 3. chrF++ / BLEU comparison


In [5]:
chrf=CHRF(word_order=2);bleu=BLEU(effective_order=True);metrics=[]
conditions=[('all','all')]+[('all',l) for l in ['en','ko']]+[(s,l) for s in sorted(pred_df.source.unique()) for l in ['en','ko']]
for source,lang in conditions:
    g=pred_df
    if source!='all':g=g[g.source.eq(source)]
    if lang!='all':g=g[g.language.eq(lang)]
    refs=g.reference.tolist();gp=g.greedy_translation.tolist();npred=g.nbest_translation.tolist()
    gch=chrf.corpus_score(gp,[refs]).score;nch=chrf.corpus_score(npred,[refs]).score
    metrics.append({'source':source,'language':lang,'samples':len(g),'greedy_chrf_pp':gch,'nbest_chrf_pp':nch,'nbest_gain':nch-gch,
                    'greedy_bleu':bleu.corpus_score(gp,[refs]).score,'nbest_bleu':bleu.corpus_score(npred,[refs]).score,
                    'greedy_empty_rate':float(np.mean([not x.strip() for x in gp])),'nbest_empty_rate':float(np.mean([not x.strip() for x in npred]))})
metric_df=pd.DataFrame(metrics);display(metric_df);metric_df.to_csv(OUT_DIR/'metrics.csv',index=False)
display(pred_df.groupby(['source','language'],group_keys=False).head(3))

summary=[]
for lang in ['en','ko']:
    r=metric_df.query("source=='all' and language==@lang").iloc[0]
    winner='nbest' if r.nbest_chrf_pp>=r.greedy_chrf_pp else 'greedy'
    summary.append({'language':lang,'winner':winner,'best_chrf_pp':max(r.greedy_chrf_pp,r.nbest_chrf_pp),
                    'best_bleu':r.nbest_bleu if winner=='nbest' else r.greedy_bleu,'nbest_gain':r.nbest_gain})
summary_df=pd.DataFrame(summary);display(summary_df);summary_df.to_csv(OUT_DIR/'summary.csv',index=False)


,source,language,samples,greedy_chrf_pp,nbest_chrf_pp,nbest_gain,greedy_bleu,nbest_bleu,greedy_empty_rate,nbest_empty_rate
0,all,all,600,20.508566,19.902992,-0.605575,5.279816,4.857359,0.0,0.0
1,all,en,300,25.152121,24.550690,-0.601431,7.249749,6.679568,0.0,0.0
2,all,ko,300,10.441507,9.930233,-0.511274,1.361129,1.961991,0.0,0.0
3,fleurs,en,100,26.917302,26.369066,-0.548236,6.846815,6.057246,0.0,0.0
4,fleurs,ko,100,10.547147,9.646148,-0.900999,1.110253,1.251295,0.0,0.0
5,mig-burmese-audio-transcription,en,100,21.049007,19.954788,-1.094218,10.636265,8.347843,0.0,0.0
6,mig-burmese-audio-transcription,ko,100,8.624547,8.948863,0.324316,3.057123,2.868561,0.0,0.0
7,openslr,en,100,24.955972,24.561193,-0.394779,5.407158,5.892225,0.0,0.0
8,openslr,ko,100,11.464117,11.120275,-0.343842,1.216051,1.874383,0.0,0.0


,sample_id,source,language,reference,my_text,greedy_text,greedy_translation,nbest_translation
0,train_001180,fleurs,en,Xinhua reported that government investigators ...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ္ဓဟူးနေ့၏ ပျံသ...,အစိုးရဆုံစ်း စစ်ဆေးသူများသည် ဘုက�တ�ဟုနှစ်၏ ပြန...,The customs inspectors recovered two bags that...,The Royal Commission investigators were able t...
1,train_001180,fleurs,ko,신화(Xinhua)통신은 정부 조사관들이 비행기록 장치 '블랙박스' 2개를 수요일에...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ္ဓဟူးနေ့၏ ပျံသ...,အစိုးရဆုံစ်း စစ်ဆေးသူများသည် ဘုက�တ�ဟုနှစ်၏ ပြန...,샌우는 행정 검토관들이 백팩 두 개를 벌레잡이로 쓴 뒤의 귀환 여정의 기록에 복구했...,정부 감사관들은 빅토리아의 귀환 여정을 기록한 아버지의 편지를 회수할 수 있었다고 ...
2,train_002169,fleurs,en,A power failure following a routine fire-comma...,ပုံမှန် မီးငြှိမ်းသတ်ရေးဆိုင်ရာညွှန်ကြားမှုစနစ...,ပုံမှန်မီးကြိမးသက်ရေးဆိုင်ရာ ညွှန်ကြားမှုစ�န�စ...,The regular firefighting command sequence was ...,Regular firefighting operations were interrupt...
3,train_002169,fleurs,ko,"정기적인 화재 지휘 시스템 시험 뒤에 정전이 발생해 압력 방출 밸브가 열렸고, 포트...",ပုံမှန် မီးငြှိမ်းသတ်ရေးဆိုင်ရာညွှန်ကြားမှုစနစ...,ပုံမှန်မီးကြိမးသက်ရေးဆိုင်ရာ ညွှန်ကြားမှုစ�န�စ...,"정기적인 불꽃 감시 점검이 중단된 뒤, 수원은 몇 분 안에 사라졌고 수도는 끊겼다.",정기적인 연료전선 관리가 우기야리 쪽의 폭발로 인해 성층간 구름이 흩어져 전력망이 ...
4,train_001500,fleurs,en,"""Apart from the fever and a sore throat, I fee...",“အဖျားနှင့် လည်ချောင်းနာသည်လွဲ၍ နေအိမ်မှ ဆက်သွ...,အဖျားနှင့်လ်ကောင်းနာသည်လွဲ၍ နေမဆက်သွယ် လုပ်ဆော...,"Abe and Nakano are mostly just good company, a...",The only thing that is different from the feve...
5,train_001500,fleurs,ko,"열과 목의 통증을 빼면, 저는 컨디션은 괜찮고 재택 근무로 제 일을 할 수 있을 정...",“အဖျားနှင့် လည်ချောင်းနာသည်လွဲ၍ နေအိမ်မှ ဆက်သွ...,အဖျားနှင့်လ်ကောင်းနာသည်လွဲ၍ နေမဆက်သွယ် လုပ်ဆော...,"그는 아프지만, 그 밖에는 아무것도 없었습니다.","열이 나고 복통이 심은 상태였지만, 그 밖에는 아무것도 없었다."
200,train_006319,mig-burmese-audio-transcription,en,Calculating that they could make us dance to t...,ငါတို့ စိတ်တိုင်းကျ ကြိုးဆွဲရာ ကခိုင်းလို့ ရမယ...,�ကတို့ စိတ်�န်း� ကိုးဆ�်ရာ ကခို်းို� ရာမယ�လို့...,They calculated that the journey from Kawkarei...,"They calculated it as 'Ko, Thit Nyein, Ko, Sab..."
201,train_006319,mig-burmese-audio-transcription,ko,우리를 자기들 뜻대로 움직일 수 있다고 계산하며…,ငါတို့ စိတ်တိုင်းကျ ကြိုးဆွဲရာ ကခိုင်းလို့ ရမယ...,�ကတို့ စိတ်�န်း� ကိုးဆ�်ရာ ကခို်းို� ရာမယ�လို့...,그들은 ‘우리 약속을 지키고 싶다’는 말을 들은 쏘 론이 무릎 꿇었다.,그들은 '아홉 번째와 열 번째의 차이는 그저 마음의 수치로 쓰인다'고 계산했다.
202,train_006907,mig-burmese-audio-transcription,en,But in reality...,ဒါပေမဲ့ တကယ်မှာကကျတော့,ဒါပမ�့ တကယ်မှာကျတော့,In fact...,In fact...
203,train_006907,mig-burmese-audio-transcription,ko,하지만 실제로는…,ဒါပေမဲ့ တကယ်မှာကကျတော့,ဒါပမ�့ တကယ်မှာကျတော့,사실은...,사실은...


,language,winner,best_chrf_pp,best_bleu,nbest_gain
0,en,greedy,25.152121,7.249749,-0.601431
1,ko,greedy,10.441507,1.361129,-0.511274


## 4. Compare against previous zero-shot and Oracle numbers


In [6]:
known=[]
for lang,zero_greedy,zero_nbest,oracle_before,oracle_lora in [
    ('en',22.242002,23.746007,33.314048,44.980627),
    ('ko',8.445128,8.444171,13.536367,22.797998),
]:
    now=metric_df.query("source=='all' and language==@lang").iloc[0]
    known.extend([
        {'language':lang,'condition':'zero-shot CTC greedy','chrf_pp':zero_greedy},
        {'language':lang,'condition':'zero-shot CTC N-best','chrf_pp':zero_nbest},
        {'language':lang,'condition':'zero-shot Oracle','chrf_pp':oracle_before},
        {'language':lang,'condition':'fine-tuned Oracle','chrf_pp':oracle_lora},
        {'language':lang,'condition':'fine-tuned CTC greedy','chrf_pp':now.greedy_chrf_pp},
        {'language':lang,'condition':'fine-tuned CTC N-best','chrf_pp':now.nbest_chrf_pp},
    ])
comparison=pd.DataFrame(known);display(comparison);comparison.to_csv(OUT_DIR/'comparison_with_baselines.csv',index=False)


,language,condition,chrf_pp
0,en,zero-shot CTC greedy,22.242002
1,en,zero-shot CTC N-best,23.746007
2,en,zero-shot Oracle,33.314048
3,en,fine-tuned Oracle,44.980627
4,en,fine-tuned CTC greedy,25.152121
5,en,fine-tuned CTC N-best,24.550690
6,ko,zero-shot CTC greedy,8.445128
7,ko,zero-shot CTC N-best,8.444171
8,ko,zero-shot Oracle,13.536367
9,ko,fine-tuned Oracle,22.797998


## 판정

- Fine-tuned CTC 점수가 zero-shot CTC보다 크게 오르면 clean LoRA가 noisy input에도 일반화한 것입니다.
- N-best가 greedy보다 높으면 N-best prompt를 유지합니다.
- Fine-tuned Oracle과 CTC 사이의 차이가 크면 다음 단계에서만 noisy adaptation을 수행합니다.
- 이 노트북은 평가 전용이며 어떤 adapter도 수정하지 않습니다.
